<a href='https://colab.research.google.com/github/Emelecto/QuantLab/blob/main/web/content/cursos/fundamentos/notebooks/c1_l6.ipynb' target='_parent'><img src='https://colab.research.google.com/assets/colab-badge.svg'/></a>

# C1-L6 · Auditoría de 20 variantes SMA
Audita un cruce de medias: separa dentro (IS) y fuera (OOS) de muestra y mide el gap. Pandas + matplotlib.

In [ ]:
import pandas as pd
from pathlib import Path

CSV = 'c1_l6_sma20.csv'
URL = 'https://raw.githubusercontent.com/Emelecto/QuantLab/main/web/content/cursos/fundamentos/data/' + CSV
try:
    df = pd.read_csv(URL)
    print('Fuente: URL (Colab)')
except Exception as e:
    print('Sin red, uso fallback local:', e)
    for cand in [Path('../data') / CSV, Path('data') / CSV, Path(CSV)]:
        if cand.exists():
            df = pd.read_csv(cand)
            break
    print('Fuente: local')
print('variantes:', len(df), ' columnas:', list(df.columns))
print(df['veredicto'].value_counts().to_string())
print(df.head(3).to_string(index=False))

In [ ]:
best_is = df.loc[df['sharpe_is'].idxmax()]
print(f"campeona IS: ({int(best_is['fast'])},{int(best_is['slow'])})  IS={best_is['sharpe_is']:.3f}  OOS={best_is['sharpe_oos']:.3f}  gap={best_is['gap']:.3f} → {best_is['veredicto']}")
peor_gap = df.loc[df['gap'].abs().idxmax()]
print(f"mayor |gap|: ({int(peor_gap['fast'])},{int(peor_gap['slow'])})  gap={peor_gap['gap']:.3f} → {peor_gap['veredicto']}")
print(f"gap medio robustas: {df.loc[df['veredicto']=='robusta','gap'].abs().mean():.3f}")
print(f"gap medio no robustas: {df.loc[df['veredicto']!='robusta','gap'].abs().mean():.3f}")

## Robustez = OOS decente + gap chico
La campeona en IS rara vez repite. Grafica IS vs OOS: lo robusto vive pegado a la diagonal.

In [ ]:
import matplotlib.pyplot as plt

color = {'robusta': '#5eead4', 'debil': '#8a8a93', 'sobreajustada': '#f59e0b'}
fig, ax = plt.subplots(figsize=(6, 4))
for v, g in df.groupby('veredicto'):
    ax.scatter(g['sharpe_is'], g['sharpe_oos'], c=color[v], label=v, s=60)
lims = [df[['sharpe_is','sharpe_oos']].min().min() - 0.3, df[['sharpe_is','sharpe_oos']].max().max() + 0.3]
ax.plot(lims, lims, '--', c='#52525b', label='diagonal (gap=0)')
ax.set_xlabel('Sharpe IS')
ax.set_ylabel('Sharpe OOS')
ax.set_title('IS vs OOS: lo robusto vive en la diagonal')
ax.legend()
fig.tight_layout()
fig

In [ ]:
# Chequeos automáticos
assert len(df) == 20, 'debe haber 20 variantes'
assert (df['veredicto'].value_counts()['robusta'] == 9)
assert (df['veredicto'].value_counts()['debil'] == 10)
assert (df['veredicto'].value_counts()['sobreajustada'] == 1)
assert int(best_is['fast']) == 10 and int(best_is['slow']) == 30, 'la campeona IS debe ser (10,30)'
assert best_is['veredicto'] != 'robusta', 'la campeona IS no es robusta: esa es la lección'
fila_418 = df[(df['fast'] == 4) & (df['slow'] == 18)].iloc[0]
assert abs(fila_418['gap']) < 0.01, 'la (4,18) debe tener gap ~0'
print('OK: 20 variantes auditadas, la campeona IS no es robusta')